# HydroAlert: Station-Anchored Multimodal Flash-Flood Warning Pipeline Walkthrough

This interactive notebook demonstrates the end-to-end pipeline execution from NOAA GHCNh ingestion to downhill DEM flow-routing alerts and ablation benchmarking.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.pipeline_runner import FlashFloodPipelineRunner
from src.ingestion.ghcnh_parser import GHCNhParser
from src.terrain.dem_pipeline import DEMPipeline
from src.trajectory.manifest import MissingFrameManifest
from src.model.multimodal_model import MultimodalFlashFloodNet

## 1. Run Complete Pipeline
Executes Ingestion, Trajectory Building, DEM Analysis, Multimodal Inference, Downhill Alerts, and Ablation Study.

In [2]:
runner = FlashFloodPipelineRunner()
results = runner.run_all()
print(f"Pipeline run complete! Samples processed: {results['num_samples']}")

## 2. Inspect Satellite Trajectory Manifest & Missing-Frame Handling (NFR1)
Validates that missing frames are explicitly recorded and never forward-filled.

In [3]:
cov = runner.manifest.compute_coverage_statistics()
print(f"Total frames: {cov['total_frames']}")
print(f"Valid frames ('ok'): {cov['ok_frames']} ({cov['coverage_pct']}%)")
print(f"Missing frames: {cov['missing_frames']}")

is_valid, violations = runner.manifest.verify_nfr1_integrity()
print(f"NFR1 Integrity Audit: {'PASSED' if is_valid else 'FAILED'}")

## 3. Compare Alert Methods: Circular Buffer (v0) vs DEM Flow Routing (v1)
Notice how circular buffer triggers false alarms on uphill peaks, while DEM flow routing follows gravity downhill.

In [4]:
summary = results['alert_summary']
print("Circular Buffer (v0):", summary['v0_circular_buffer'])
print("DEM Flow Routing (v1):", summary['v1_dem_flow_routing'])

## 4. Ablation Benchmark Results

In [5]:
ablation = results['ablation_results']
for k, v in ablation.items():
    print(f"{v['stage_name']:<42} | CSI: {v['csi']:.3f} | POD: {v['pod']:.3f} | FAR: {v['far']:.3f}")